In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from config import (
    N, p0, Lambda, lam, T, y_intervals,
    t_net_filtering, pi_uniform, pi_uniform, M_net,
    ht, delta, F, G, get_obs, exp_id
)
from SMJP import (
    sparse_mc, get_y_uniform, 
    make_discretized_xi, make_discretized_eta
)
from utils import load_saved_path, save_path
from filter import Filter
from plots import plot_theta, plot_y, plot_obs

In [ ]:
load_path = False

In [ ]:
if load_path:
    theta, y, t, theta_est, y_est, dxi, deta = load_saved_path(exp_id)
else:
    theta, y, t = sparse_mc(p0, Lambda, lam, T, get_y_uniform, y_intervals)
    observations = get_obs(t_net_filtering, theta, y, t)

In [ ]:
if not load_path:
    filter = Filter(
        p0[:, np.newaxis] * pi_uniform, 
        pi_uniform, M_net, F, G,
        N, Lambda, ht, delta
    )

    est = filter.estimate()
    theta_est = [est[0]]
    y_est = [est[1]]

    for i, obs in enumerate(tqdm(observations.squeeze()), start=1):
        #obs = np.array([obs])
        filter.update(obs)
        est = filter.estimate()
        if np.any(np.isnan(est[0])) or np.any(np.isnan(est[1])):
            print(f'nan on {i}-th iter')
            break
        theta_est.append(est[0])
        y_est.append(est[1])
    
    theta_est = np.array(theta_est)
    y_est = np.array(y_est)

In [ ]:
fig, ax = plot_theta(theta, t, theta_est)

plt.show()

In [ ]:
save_path(exp_id, theta, y, t, theta_est, y_est, observations)

In [ ]:
fig, ax = plot_y(theta, y, t, y_est)

plt.show()

In [ ]:
# with open('saved_path/theta_est.pkl', 'wb') as f:
#     pickle.dump(theta_est, f)
# with open('saved_path/y_est.pkl', 'wb') as f:
#     pickle.dump(y_est, f)
# with open('saved_path/theta.pkl', 'wb') as f:
#     pickle.dump(theta, f)
# with open('saved_path/y.pkl', 'wb') as f:
#     pickle.dump(y, f)
# with open('saved_path/t.pkl', 'wb') as f:
#     pickle.dump(t, f)